# CenterLoRA-ProtBERT v1 — frozen full-data refit and independent test

This is the final seed-42 evaluation. The architecture and hyperparameters are frozen from development. It refits a fresh model for the development-selected **4 epochs** on all 91,723 released training records, then provides a separate, explicit one-time evaluation on the 12,598-record independent test. The primary threshold is 0.5; 0.555 is a secondary threshold selected before test access. Select a GPU runtime.

Running the final evaluation closes architecture exploration. Do not change the architecture, epoch count, seed, or thresholds after viewing the independent-test result.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update the private repository

Add a fine-grained read-only GitHub token to Colab Secrets as `GITHUB_TOKEN`. Optionally add `HF_TOKEN` for authenticated Hugging Face downloads. Enable notebook access for each secret.

In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
    try:
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GITHUB_TOKEN = None
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        HF_TOKEN = None
except ImportError:
    GITHUB_TOKEN = None
    HF_TOKEN = None

git_environment = os.environ.copy()
git_environment['GIT_TERMINAL_PROMPT'] = '0'
askpass_path = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass_path.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding='utf-8',
    )
    askpass_path.chmod(0o700)
    git_environment['GIT_ASKPASS'] = str(askpass_path)
    git_environment['GITHUB_TOKEN'] = GITHUB_TOKEN

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run([
            'git', 'clone', '--recurse-submodules',
            'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git',
            str(REPO_ROOT),
        ], check=True, env=git_environment)
    else:
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'],
            check=True, env=git_environment,
        )
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'],
            check=True, env=git_environment,
        )
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Git access failed. Add GITHUB_TOKEN to Colab Secrets for a private repository.'
    ) from error
finally:
    askpass_path.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HF_HOME'] = str(
    Path('/content/drive/MyDrive/MMUbiPred-research/huggingface-cache')
)
Path(os.environ['HF_HOME']).mkdir(parents=True, exist_ok=True)
print('Project:', REPO_ROOT)
print('Persistent Hugging Face cache:', os.environ['HF_HOME'])
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt
%pip uninstall -y torchao

In [ ]:
import json
import platform
import sys
import peft
import sklearn
import torch
import transformers

print('Python:', sys.version)
print('Platform:', platform.platform())
print('scikit-learn:', sklearn.__version__)
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('PEFT:', peft.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    properties = torch.cuda.get_device_properties(0)
    print('GPU:', properties.name)
    print('GPU memory (GiB):', round(properties.total_memory / 2**30, 2))
else:
    raise RuntimeError('Select a GPU runtime before starting the final refit.')

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError('Could not open subprocess output stream')
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## Preflight tests

In [ ]:
run_live([
    sys.executable, '-u', '-m', 'unittest', 'discover',
    '-s', 'experiment/tests',
])

## Verify the frozen development selection

The complete seed-42 development folder copied earlier must remain in Drive. The refit script checks its experiment name, model configuration, best epoch, selected threshold, non-collapse status, and validation hash.

In [ ]:
SELECTION_RUN_DIR = Path(
    '/content/drive/MyDrive/MMUbiPred-research/experiments/'
    'center-lora-protbert-v1-seed42'
)
required_selection_files = [
    SELECTION_RUN_DIR / 'run_manifest.json',
    SELECTION_RUN_DIR / 'validation_metrics.json',
]
missing = [str(path) for path in required_selection_files if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing frozen development artifacts: {missing}')
print('Frozen development selection:', SELECTION_RUN_DIR)

## Run or resume the four-epoch full-data refit

This cell reads only the released training FASTA files. It saves an epoch-boundary resume checkpoint to Drive. If Colab disconnects, rerun the notebook and this cell; an unfinished epoch is repeated, while completed epochs are preserved.

In [ ]:
FINAL_RUN_DIR = Path(
    '/content/drive/MyDrive/MMUbiPred-research/final/'
    'center-lora-protbert-v1-seed42-full-train'
)
FINAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
REFIT_SUMMARY_PATH = FINAL_RUN_DIR / 'refit_summary.json'
REFIT_RESUME_PATH = FINAL_RUN_DIR / 'refit_resume.pt'

if REFIT_SUMMARY_PATH.exists() and (FINAL_RUN_DIR / 'best.pt').exists():
    print('Completed full-data refit already exists; training was skipped.')
else:
    if REFIT_RESUME_PATH.exists():
        resume_state = torch.load(
            REFIT_RESUME_PATH, map_location='cpu', weights_only=False
        )
        print('Resuming after completed refit epoch:', resume_state['completed_epoch'])
        del resume_state
    run_live([
        sys.executable, '-u',
        'experiment/scripts/refit_frozen_candidate.py',
        '--config',
        'experiment/configs/center_lora_protbert_final_seed42.json',
        '--selection-run-dir', SELECTION_RUN_DIR,
        '--output-dir', FINAL_RUN_DIR,
        '--resume',
    ])

refit_summary = json.loads(REFIT_SUMMARY_PATH.read_text())
refit_manifest = json.loads((FINAL_RUN_DIR / 'run_manifest.json').read_text())
display(refit_summary)
display({
    'complete_training_set': refit_manifest['complete_training_set'],
    'selection_provenance': refit_manifest['selection_provenance'],
    'final_refit': refit_manifest['final_refit'],
    'locked_test_accessed': refit_manifest['locked_test_accessed'],
})

## One-time independent-test evaluation

This is the irreversible test-access step. Confirm that the refit completed with 91,723 samples and 4 epochs. Change the acknowledgement to `True` only when ready. The evaluator refuses to overwrite an existing locked-test result.

In [ ]:
ACKNOWLEDGE_FINAL_LOCKED_TEST = False
LOCKED_METRICS_PATH = FINAL_RUN_DIR / 'locked_test_metrics.json'

if LOCKED_METRICS_PATH.exists():
    print('Existing one-time independent-test result was not rerun.')
    locked_metrics = json.loads(LOCKED_METRICS_PATH.read_text())
    display(locked_metrics)
elif ACKNOWLEDGE_FINAL_LOCKED_TEST:
    run_live([
        sys.executable, '-u', 'experiment/scripts/evaluate.py',
        '--run-dir', FINAL_RUN_DIR,
        '--data-dir', 'replication/MMUbiPred',
        '--batch-size', '32',
        '--num-workers', '0',
        '--require-full-refit',
        '--allow-locked-test',
    ])
    locked_metrics = json.loads(LOCKED_METRICS_PATH.read_text())
    display(locked_metrics)
else:
    print('Locked test was not accessed. Set ACKNOWLEDGE_FINAL_LOCKED_TEST=True when ready.')

## Primary comparison with the reproduced paper model

In [ ]:
if LOCKED_METRICS_PATH.exists():
    locked_metrics = json.loads(LOCKED_METRICS_PATH.read_text())
    display({
        'center_lora_protbert_fixed_0.5': locked_metrics['fixed_threshold'],
        'center_lora_protbert_dev_selected_0.555': locked_metrics['validation_selected_threshold'],
        'reproduced_paper_baseline': locked_metrics['paper_baseline'],
        'fixed_0.5_difference_from_paper': locked_metrics['fixed_threshold_difference_from_paper'],
    })
else:
    print('Run the acknowledged one-time evaluation before displaying final results.')